**Устанавливаем библиотеки**

In [ ]:
# !pip install sqlalchemy
# !pip install psycopg2
# !pip install requests

**Импорт библиотек. Настройка API**

In [5]:
import requests
import json
import re
import psycopg2
import pandas as pd
import time
from sqlalchemy import create_engine
from typing import Dict, List, Optional

# КОНФИГУРАЦИЯ
# ключ для бесплатных LLM на Openrouter
OPENROUTER_API_KEY = "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"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
# Пока тестируем на бесплатных LLM с хорошим рейтингом по программированию
GENERATOR_MODEL = "openrouter/owl-alpha"
FIXER_MODEL = "openrouter/owl-alpha"

**Определяем правила для генератора**

In [6]:
def generate_sql(
    user_query: str,
    schema_description: str,
    rules: List[str],
    model: str = GENERATOR_MODEL
) -> Dict:
    """
    Отправляет запрос в LLM и получает JSON с SQL и списком таблиц.

    Параметры:
        user_query: текстовое описание задачи пользователя
        schema_description: описание схемы БД (таблицы, поля, связи)
        rules: список текстовых правил (без SELECT *, WHERE для фильтрации и т.п.)
        model: идентификатор модели на OpenRouter

    Возвращает:
        dict с ключами "sql" и "tables_used"
    """
    # Формируем системный промпт
    system_prompt = (
        "Ты — SQL-ассистент, который генерирует запросы к PostgreSQL.\n"
        "Всегда возвращай ответ строго в JSON-формате без каких-либо дополнительных комментариев:\n"
        '{"sql": "<стока SQL-запроса>", "tables_used": ["<таблица1>", "<таблица2>"]}\n\n'
        "Правила, которые необходимо соблюдать:\n" +
        "\n".join(f"- {rule}" for rule in rules) +
        "\n\nСхема базы данных:\n" + schema_description
    )

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        "temperature": 0.1,  # низкая температура для детерминированности
        "response_format": {"type": "json_object"}  # принудительный JSON для поддерживающих моделей
    }

    # Будем считать время генерации ответа генератора
    start_time = time.time()

    # Запрос к LLM
    response = requests.post(OPENROUTER_URL, headers=headers, json=payload)
    response.raise_for_status()  # выбросит исключение при HTTP-ошибке

    result = response.json()
    content = result["choices"][0]["message"]["content"]

    # Также проверим сколько токенов скушал генератор
    usage = result.get("usage", {})
    tokens_used = {
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "total_tokens": usage.get("total_tokens", 0)
    }
    execution_time = time.time() - start_time

    # Пытаемся распарсить JSON из ответа
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        # Если модель оборачивает JSON в markdown-блоки, убираем их
        match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", content, re.DOTALL)
        if match:
            parsed = json.loads(match.group(1))
        else:
            raise ValueError(f"Не удалось распарсить JSON из ответа: {content}")

    # Проверяем наличие обязательных полей
    if "sql" not in parsed or "tables_used" not in parsed:
        raise ValueError(f"Ответ модели не содержит необходимые поля 'sql' и 'tables_used': {parsed}")

    parsed["tokens_used"] = tokens_used
    parsed["execution_time_seconds"] = round(execution_time, 3)

    return parsed

**Определяем правила для Fixer**

In [7]:
def fix_sql(
    current_sql: str,
    violations: List[Dict],  # список словарей с описанием нарушений
    schema_description: str,
    rules: List[str],
    model: str = FIXER_MODEL
) -> Dict:
    """
    Исправляет SQL-запрос на основе списка нарушений.

    Параметры:
        current_sql: текущий SQL-запрос
        violations: список нарушений, например [{"line": 1, "column": 10, "rule": "no SELECT *", "description": "..."}]
        schema_description: описание схемы БД
        rules: список правил
        model: модель для исправления (может отличаться от генератора)

    Возвращает:
        {"sql": "<исправленный SQL>", "changes": "<описание изменений>"}
    """
    # Формируем список нарушений в текстовом виде
    violations_text = []
    for i, v in enumerate(violations, 1):
        violations_text.append(
            f"{i}. Строка {v.get('line', '?')}, колонка {v.get('column', '?')}: "
            f"{v.get('rule', 'нарушение')} - {v.get('description', '')}"
        )
    violations_str = "\n".join(violations_text)

    system_prompt = (
        "Ты — эксперт по безопасному SQL. Твоя задача — исправить указанные нарушения в существующем SQL-запросе.\n"
        "Меняй **только** те части запроса, которые связаны с перечисленными нарушениями. "
        "Не изменяй логику запроса и не вноси исправления там, где замечаний нет.\n"
        "Всегда возвращай ответ строго в JSON-формате без комментариев:\n"
        '{"sql": "<строка исправленного SQL>", "changes": "<краткое описание внесённых изменений>"}\n\n'
        "Общие правила:\n" + "\n".join(f"- {rule}" for rule in rules) +
        "\n\nСхема БД:\n" + schema_description
    )

    user_prompt = (
        f"Текущий SQL-запрос:\n```sql\n{current_sql}\n```\n\n"
        f"Нарушения, которые нужно исправить:\n{violations_str}\n\n"
        "Исправь запрос, следуя указаниям."
    )

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.1
    }

    # Будем считать время генерации ответа фиксера
    start_time = time.time()

    # Запрос к LLM
    response = requests.post(OPENROUTER_URL, headers=headers, json=payload)
    response.raise_for_status()

    result = response.json()
    content = result["choices"][0]["message"]["content"]

    # Также проверим сколько токенов скушал фиксер
    usage = result.get("usage", {})
    tokens_used = {
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "total_tokens": usage.get("total_tokens", 0)
    }
    execution_time = time.time() - start_time


    # Парсинг JSON
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError:
        match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", content, re.DOTALL)
        if match:
            parsed = json.loads(match.group(1))
        else:
            raise ValueError(f"Не удалось распарсить JSON: {content}")

    if "sql" not in parsed or "changes" not in parsed:
        raise ValueError(f"Отсутствуют поля 'sql' и/или 'changes': {parsed}")

    parsed["tokens_used"] = tokens_used
    parsed["execution_time_seconds"] = round(execution_time, 3)

    return parsed


**Тестовый стенд: схема БД, запрос пользователя, формирование JSON файла от генератора, обработка фиксером JSON файла от судьи**

In [11]:
if __name__ == "__main__":
    # Тест генератора
    schema = """
    Таблица users:
    - id SERIAL PRIMARY KEY
    - name VARCHAR(100)
    - email VARCHAR(150)
    - created_at TIMESTAMP

    Таблица orders:
    - id SERIAL PRIMARY KEY
    - user_id INTEGER REFERENCES users(id)
    - total_amount DECIMAL(10,2)
    - status VARCHAR(20)
    - order_date DATE
    """
    rules = [
        "no SELECT *",
        "must use WHERE for any filtering",
        "only allowed tables: users, orders",
        "max LIMIT 1000"
    ]
    # Запрос от пользователя. Пока здесь
    user_query = "Покажи все заказы пользователя с именем 'Белякова Раиса Андреевна'"

    print("*** Проверка генерации ***")
    gen_result = generate_sql(user_query, schema, rules)

    # Сохраняем результат в JSON-файл
    with open("generated_sql.json", "w", encoding="utf-8") as f:
        json.dump(gen_result, f, indent=2, ensure_ascii=False)

    print("SQL:", gen_result["sql"])
    print("Tables:", gen_result["tables_used"])

    # Имитируем нарушения от Security Judge
    # fake_violations = [
    #     {
    #         "line": 1,
    #         "column": 8,
    #         "rule": "no SELECT *",
    #         "description": "Запрос содержит SELECT *, что увеличивает нагрузку и небезопасно."
    #     }
    # ]

    # Загрузка нарушений от Security Judge из JSON файла
    with open("violations.json", "r", encoding="utf-8") as f:
        fake_violations = json.load(f)

    # Допустим, генератор случайно вернул SELECT *
    # bad_sql = "SELECT * FROM users JOIN orders ON users.id = orders.user_id WHERE orders.status = 'delivered';"

    # Загрузка некорректного SQL из JSON файла. Сейчас это ручной файл, в дальнейшем это будет JSON ответ от судьи
    with open("bad_generated_sql.json", "r", encoding="utf-8") as bad_req:
        gen_data = json.load(bad_req)
    bad_sql = gen_data["sql"]

    print("\n*** Некорректный запрос ***")
    print(bad_sql)
    print("\n*** Исправление ***")
    fix_result = fix_sql(bad_sql, fake_violations, schema, rules)
    print("Исправленный SQL:", fix_result["sql"])
    print("Изменения:", fix_result["changes"])

    # Вывод статистики
    print("\n*** Статистика ***")
    print(f"Генерация выполнена за {gen_result['execution_time_seconds']:.2f} сек.")
    print(f"Потрачено токенов генератором: {gen_result['tokens_used']['total_tokens']} "
      f"(промт: {gen_result['tokens_used']['prompt_tokens']}, "
      f"ответ: {gen_result['tokens_used']['completion_tokens']})")

    print(f"\nИсправление выполнено за {fix_result['execution_time_seconds']:.2f} сек.")
    print(f"Потрачено токенов фиксером: {fix_result['tokens_used']['total_tokens']} "
      f"(промт: {fix_result['tokens_used']['prompt_tokens']}, "
      f"ответ: {fix_result['tokens_used']['completion_tokens']})")

*** Проверка генерации ***
SQL: SELECT o.id, o.total_amount, o.status, o.order_date FROM orders o JOIN users u ON o.user_id = u.id WHERE u.name = 'Белякова Раиса Андреевна' LIMIT 1000;
Tables: ['orders', 'users']

*** Некорректный запрос ***
SELECT * FROM users JOIN orders ON users.id = orders.user_id WHERE orders.status = 'delivered'

*** Исправление ***
Исправленный SQL: SELECT users.id, users.name, users.email, users.created_at, orders.id, orders.user_id, orders.total_amount, orders.status, orders.order_date FROM users JOIN orders ON users.id = orders.user_id WHERE orders.status = 'delivered' LIMIT 1000
Изменения: Заменил SELECT * на явное перечисление всех столбцов из таблиц users и orders, а также добавил LIMIT 1000 для ограничения количества возвращаемых строк.

*** Статистика ***
Генерация выполнена за 4.21 сек.
Потрачено токенов генератором: 431 (промт: 363, ответ: 68)

Исправление выполнено за 7.93 сек.
Потрачено токенов фиксером: 667 (промт: 530, ответ: 137)


**Проверка исправленного запроса на тестовой БД**

In [ ]:
# ================= НАСТРОЙКИ ПОДКЛЮЧЕНИЯ =================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres1",     # тестовая база
    "user": "postgres",
    "password": "postgres"
}
# ==========================================================

# Подключение для SQLAlchemy
db_url = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"

try:
    engine = create_engine(db_url)
    df = pd.read_sql_query(fix_result["sql"], engine)
    print(f"Запрос вернул {len(df)} строк. Первые 20 строк:")
    display(df.head(20))
except Exception as e:
    print(f"Ошибка при выполнении запроса: {e}")

Запрос вернул 23 строк. Первые 20 строк:


,id,name,email,created_at,id,user_id,total_amount,status,order_date
0,11,Белякова Раиса Андреевна,белякова.раиса.андреевна2446@example.com,2025-03-06 08:51:19,13,11,7054.88,delivered,2025-08-15
1,41,тов. Федорова Синклитикия Леоновна,тов..федорова.синклитикия.леоновна9082@example...,2026-03-28 08:33:57,82,41,1911.84,delivered,2026-02-14
2,125,Ерофей Фролович Григорьев,ерофей.фролович.григорьев8866@example.com,2024-08-14 00:01:19,65,125,4774.55,delivered,2025-11-12
3,149,г-н Щербаков Чеслав Трофимович,г-н.щербаков.чеслав.трофимович935@example.com,2024-08-16 23:06:52,96,149,352.48,delivered,2026-04-14
4,170,Сидорова Жанна Олеговна,сидорова.жанна.олеговна3792@example.com,2026-03-20 07:31:11,18,170,3848.50,delivered,2026-04-02
5,217,Парамон Харитонович Зайцев,парамон.харитонович.зайцев6540@example.com,2024-08-30 06:18:43,1,217,8796.34,delivered,2025-10-23
6,250,Михеев Никифор Антонович,михеев.никифор.антонович83@example.com,2024-08-30 05:14:57,70,250,5908.20,delivered,2025-11-22
7,346,Порфирий Денисович Тетерин,порфирий.денисович.тетерин4145@example.com,2025-03-29 00:57:11,61,346,9056.67,delivered,2026-03-19
8,355,Ираида Даниловна Турова,ираида.даниловна.турова7804@example.com,2026-04-16 12:58:32,86,355,2337.75,delivered,2026-02-27
9,358,Нина Владимировна Мясникова,нина.владимировна.мясникова6710@example.com,2025-03-17 12:53:42,79,358,551.33,delivered,2025-05-13
